# NST2062 Technical Assignment #2 (2026)
## Investigating Minds and Machines From the Inside

**Total: 100 points (80 + 20 bonus)**

**Estimated time: 4–8 hours**

Dear Friend, this assignment has two problems. Both use large language models as the **object of study** — not just as a tool. You will probe what models know, how they express (or fail to express) uncertainty, and how the reward signal shapes their outputs.

### Ground Rules
- You **may** use LLMs (ChatGPT, Claude, Copilot, etc.) to help you write code, debug, and brainstorm. This is expected.
- You **must** document every LLM interaction that materially shaped your solution in the **Metacognitive Reflection** sections. Prompts, what worked, what didn't, what you had to fix.
- The reflection sections are **graded** and cannot be generated by an LLM. They must be in your own voice.
- Submit: this completed notebook (.ipynb) with all cells run + outputs visible.

---

## Problem 1: Calibration & Confidence — Do Models Know What They Don't Know?
### (45 points + 10 bonus)

**Course connections**: Decisions Under Uncertainty (Bayesian brain, calibration, overconfidence), Representation (what's encoded in embedding space), Learning & Reward (RLHF shapes confidence).

**The question**: When an LLM says it's confident, should you believe it? You saw in the Calibration Game (Lecture 4) that humans are overconfident on hard questions and underconfident on easy ones. LLMs are overconfident uniformly — because RLHF rewarded confident-sounding answers. In this problem, you'll measure this empirically.

---

### Part A: Build a Calibration Dataset (10 points)

Create a dataset of **50 factual questions** with known ground-truth answers. Your dataset must include:
- 10 **easy** questions (most people would know: e.g., "What is the capital of France?")
- 15 **medium** questions (educated guess: e.g., "How many bones in the adult human body?")
- 15 **hard** questions (specialist knowledge: e.g., "What year was the Rescorla-Wagner model published?")
- 10 **unanswerable/trick** questions (no clear answer, or trick framing: e.g., "What is the weight of the colour blue?")

For each question, record:
- The question text
- The ground-truth answer (or "unanswerable" for trick questions)
- The difficulty category (easy / medium / hard / unanswerable)

**Tip**: Mix domains — some from the course content (neuroscience, AI), some general knowledge, some current events. The mix matters for the analysis.

In [ ]:
# Part A: Build a 50-question calibration dataset
import re
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

questions = [
    # EASY (10)
    {"question": "What is the capital of France?", "answer": "Paris", "difficulty": "easy"},
    {"question": "How many days are there in a week?", "answer": "7", "difficulty": "easy"},
    {"question": "Which planet is known as the Red Planet?", "answer": "Mars", "difficulty": "easy"},
    {"question": "What gas do plants primarily absorb from the atmosphere for photosynthesis?", "answer": "Carbon dioxide", "difficulty": "easy"},
    {"question": "Who wrote 'Romeo and Juliet'?", "answer": "William Shakespeare", "difficulty": "easy"},
    {"question": "What is the largest ocean on Earth?", "answer": "Pacific Ocean", "difficulty": "easy"},
    {"question": "What is H2O commonly called?", "answer": "Water", "difficulty": "easy"},
    {"question": "Which continent is Egypt in?", "answer": "Africa", "difficulty": "easy"},
    {"question": "What is 9 x 9?", "answer": "81", "difficulty": "easy"},
    {"question": "What instrument has keys, pedals, and strings and is often found in concert halls?", "answer": "Piano", "difficulty": "easy"},

    # MEDIUM (15)
    {"question": "How many bones are in the adult human body?", "answer": "206", "difficulty": "medium"},
    {"question": "In which year did the Berlin Wall fall?", "answer": "1989", "difficulty": "medium"},
    {"question": "What is the chemical symbol for gold?", "answer": "Au", "difficulty": "medium"},
    {"question": "Who proposed the laws of motion and universal gravitation?", "answer": "Isaac Newton", "difficulty": "medium"},
    {"question": "What is the smallest prime number?", "answer": "2", "difficulty": "medium"},
    {"question": "Which blood type is known as the universal donor for red blood cells?", "answer": "O negative", "difficulty": "medium"},
    {"question": "What does HTTP stand for?", "answer": "Hypertext Transfer Protocol", "difficulty": "medium"},
    {"question": "Which scientist is associated with natural selection?", "answer": "Charles Darwin", "difficulty": "medium"},
    {"question": "What is the capital of Canada?", "answer": "Ottawa", "difficulty": "medium"},
    {"question": "How many players are on a soccer team on the field at one time?", "answer": "11", "difficulty": "medium"},
    {"question": "Which lobe of the brain is most associated with vision?", "answer": "Occipital lobe", "difficulty": "medium"},
    {"question": "In machine learning, what does overfitting mean?", "answer": "Learning training data too specifically and generalizing poorly", "difficulty": "medium"},
    {"question": "What is the SI unit of electric current?", "answer": "Ampere", "difficulty": "medium"},
    {"question": "What does GPU stand for?", "answer": "Graphics Processing Unit", "difficulty": "medium"},
    {"question": "Which country hosted the 2016 Summer Olympics?", "answer": "Brazil", "difficulty": "medium"},

    # HARD (15)
    {"question": "In what year was the Rescorla-Wagner model published?", "answer": "1972", "difficulty": "hard"},
    {"question": "Who introduced the concept of prediction error minimization in the Free Energy Principle framework?", "answer": "Karl Friston", "difficulty": "hard"},
    {"question": "What is the main neurotransmitter released by most excitatory cortical pyramidal neurons?", "answer": "Glutamate", "difficulty": "hard"},
    {"question": "What does BOLD stand for in fMRI?", "answer": "Blood-oxygen-level-dependent", "difficulty": "hard"},
    {"question": "Which optimization algorithm uses first and second moments of gradients and is widely used in deep learning?", "answer": "Adam", "difficulty": "hard"},
    {"question": "What is the approximate wavelength range of visible light in nanometers?", "answer": "About 380 to 700 nm", "difficulty": "hard"},
    {"question": "Which theorem states that no compression algorithm can compress all possible inputs?", "answer": "No Free Lunch theorem for lossless compression", "difficulty": "hard"},
    {"question": "What is the default network in neuroscience primarily associated with?", "answer": "Internally directed cognition and self-referential thought", "difficulty": "hard"},
    {"question": "What does KL in KL divergence stand for?", "answer": "Kullback-Leibler", "difficulty": "hard"},
    {"question": "Which architecture introduced self-attention as a central mechanism in sequence modeling?", "answer": "Transformer", "difficulty": "hard"},
    {"question": "What is the asymptotic time complexity of Dijkstra's algorithm using a binary heap?", "answer": "O((V+E) log V)", "difficulty": "hard"},
    {"question": "In Bayesian terms, what is the quantity proportional to likelihood times prior?", "answer": "Posterior", "difficulty": "hard"},
    {"question": "What is the pKa of carbonic acid's first dissociation in water at room temperature (approx.)?", "answer": "About 6.35", "difficulty": "hard"},
    {"question": "Which paper introduced the term 'attention is all you need'?", "answer": "Attention Is All You Need (Vaswani et al., 2017)", "difficulty": "hard"},
    {"question": "Which metric-learning loss uses triplets of anchor, positive, and negative examples?", "answer": "Triplet loss", "difficulty": "hard"},

    # UNANSWERABLE/TRICK (10)
    {"question": "What is the exact weight of the color blue?", "answer": "unanswerable", "difficulty": "unanswerable"},
    {"question": "What did Julius Caesar eat for breakfast on his 33rd birthday?", "answer": "unanswerable", "difficulty": "unanswerable"},
    {"question": "What is the best song ever written, objectively?", "answer": "unanswerable", "difficulty": "unanswerable"},
    {"question": "How many thoughts did you have yesterday at 3:17 PM?", "answer": "unanswerable", "difficulty": "unanswerable"},
    {"question": "What is the smell of number seven in SI units?", "answer": "unanswerable", "difficulty": "unanswerable"},
    {"question": "Which came first in all possible universes, the chicken or the egg?", "answer": "unanswerable", "difficulty": "unanswerable"},
    {"question": "What will be the exact global temperature at noon 100 years from today?", "answer": "unanswerable", "difficulty": "unanswerable"},
    {"question": "What is the one true meaning of life for every human?", "answer": "unanswerable", "difficulty": "unanswerable"},
    {"question": "Which single economic policy guarantees prosperity forever?", "answer": "unanswerable", "difficulty": "unanswerable"},
    {"question": "Who was objectively the greatest thinker in all of history by universal metric?", "answer": "unanswerable", "difficulty": "unanswerable"},
]

df_questions = pd.DataFrame(questions)
assert len(df_questions) == 50, f"Expected 50 questions, got {len(df_questions)}"
print(df_questions["difficulty"].value_counts())
display(df_questions.head(10))

# Save for downstream cells
out_dir = Path("assignment_outputs")
out_dir.mkdir(exist_ok=True)
df_questions.to_csv(out_dir / "questions.csv", index=False)
print("Saved:", out_dir / "questions.csv")


def normalize_text(text: str) -> str:
    text = str(text).strip().lower()
    text = re.sub(r"[^a-z0-9\s\-\.]+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text


def is_correct_answer(model_answer: str, ground_truth: str) -> int:
    """Simple, transparent scoring: exact-ish matching + unanswerable handling."""
    gt = normalize_text(ground_truth)
    ans = normalize_text(model_answer)

    if gt == "unanswerable":
        uncertainty_markers = [
            "cannot", "cant", "can't", "unknown", "unanswerable", "no objective", "not enough",
            "no single", "depends", "impossible", "not possible", "no clear answer", "i dont know", "i don't know"
        ]
        return int(any(m in ans for m in uncertainty_markers))

    if gt in ans or ans in gt:
        return 1

    # Numeric tolerance for simple number answers
    gt_num = re.findall(r"-?\d+(?:\.\d+)?", gt)
    ans_num = re.findall(r"-?\d+(?:\.\d+)?", ans)
    if gt_num and ans_num:
        try:
            return int(abs(float(gt_num[0]) - float(ans_num[0])) < 1e-6)
        except Exception:
            pass

    # Fuzzy overlap fallback
    gt_tokens = set(gt.split())
    ans_tokens = set(ans.split())
    if gt_tokens:
        overlap = len(gt_tokens & ans_tokens) / len(gt_tokens)
        return int(overlap >= 0.6)
    return 0


def expected_calibration_error(df: pd.DataFrame, conf_col: str = "confidence", correct_col: str = "correct", n_bins: int = 10):
    tmp = df.copy()
    tmp = tmp.dropna(subset=[conf_col, correct_col])
    if tmp.empty:
        return np.nan

    tmp["conf_01"] = np.clip(tmp[conf_col] / 100.0, 0, 1)
    bins = np.linspace(0, 1, n_bins + 1).tolist()
    tmp["bin"] = pd.cut(tmp["conf_01"], bins=bins, include_lowest=True)

    ece = 0.0
    n = len(tmp)
    for _, g in tmp.groupby("bin", observed=False):
        if len(g) == 0:
            continue
        acc = g[correct_col].mean()
        conf = g["conf_01"].mean()
        ece += (len(g) / n) * abs(acc - conf)
    return ece


def calibration_by_bucket(df: pd.DataFrame, conf_col: str = "confidence", correct_col: str = "correct", n_bins: int = 10):
    tmp = df.copy().dropna(subset=[conf_col, correct_col])
    tmp["conf_01"] = np.clip(tmp[conf_col] / 100.0, 0, 1)
    bins = np.linspace(0, 1, n_bins + 1).tolist()
    tmp["bin"] = pd.cut(tmp["conf_01"], bins=bins, include_lowest=True)
    out = (
        tmp.groupby("bin", observed=False)
        .agg(
            n=(correct_col, "size"),
            accuracy=(correct_col, "mean"),
            avg_confidence=("conf_01", "mean"),
        )
        .reset_index()
    )
    out = out[out["n"] > 0].copy()
    return out


def plot_calibration_curve(df: pd.DataFrame, title: str, conf_col: str = "confidence", correct_col: str = "correct"):
    bucket = calibration_by_bucket(df, conf_col=conf_col, correct_col=correct_col, n_bins=10)
    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    if not bucket.empty:
        plt.plot(bucket["avg_confidence"], bucket["accuracy"], marker="o", label=title)
    plt.xlabel("Mean confidence (0-1)")
    plt.ylabel("Empirical accuracy")
    plt.title(title)
    plt.legend()
    plt.show()

### Part B: Measure YOUR Calibration (10 points)

Before querying any model, **answer 20 of your own questions yourself** (pick 5 from each difficulty category). For each:
1. Write your answer
2. Rate your confidence: 50% (pure guess) to 100% (certain)

Then compute:
- Your **accuracy** per difficulty category
- Your **average confidence** per difficulty category
- Your **calibration error**: |confidence - accuracy| averaged across categories

Visualise this as a **calibration plot**: x-axis = confidence bucket, y-axis = actual accuracy. A perfectly calibrated person lies on the diagonal.

**This section is personal and must be done honestly before looking at the model's answers.**

In [ ]:
# Part B: Measure YOUR calibration
# IMPORTANT: Fill this honestly before running model cells.

from IPython.display import display

if "df_questions" not in globals():
    df_questions = pd.read_csv("assignment_outputs/questions.csv")

# Sample 5 questions from each difficulty for self-evaluation (20 total)
rng_seed = 42
self_sample = (
    df_questions.groupby("difficulty", group_keys=False)
    .apply(lambda x: x.sample(5, random_state=rng_seed))
    .reset_index(drop=True)
)

# Create a template you should fill manually
self_answers_path = Path("assignment_outputs/self_answers.csv")
if self_answers_path.exists():
    self_df = pd.read_csv(self_answers_path)
else:
    self_df = self_sample.copy()
    self_df["my_answer"] = ""
    self_df["my_confidence"] = np.nan  # 50 to 100
    self_df.to_csv(self_answers_path, index=False)

print("Fill your answers in:", self_answers_path)
display(self_df.head(10))

# Re-load in case you edited the CSV externally
self_df = pd.read_csv(self_answers_path)

required_cols = {"question", "answer", "difficulty", "my_answer", "my_confidence"}
missing = required_cols - set(self_df.columns)
if missing:
    raise ValueError(f"Missing columns in self_answers.csv: {missing}")

# Compute metrics only when data is complete
if self_df["my_answer"].astype(str).str.strip().eq("").any() or self_df["my_confidence"].isna().any():
    print("Please complete all my_answer and my_confidence values (50-100) in self_answers.csv, then re-run this cell.")
else:
    self_df["my_confidence"] = self_df["my_confidence"].astype(float).clip(50, 100)
    self_df["correct"] = self_df.apply(lambda r: is_correct_answer(r["my_answer"], r["answer"]), axis=1)

    summary = (
        self_df.groupby("difficulty", as_index=False)
        .agg(
            accuracy=("correct", "mean"),
            avg_confidence=("my_confidence", "mean"),
            n=("question", "size"),
        )
    )
    summary["calibration_error_abs"] = (summary["avg_confidence"] / 100.0 - summary["accuracy"]).abs()
    overall_cal_error = summary["calibration_error_abs"].mean()

    display(summary)
    print(f"Overall calibration error (mean abs |conf-acc| across categories): {overall_cal_error:.3f}")

    plot_calibration_curve(
        self_df.rename(columns={"my_confidence": "confidence"}),
        title="Your Calibration Curve",
        conf_col="confidence",
        correct_col="correct",
    )

    self_df.to_csv("assignment_outputs/self_results_scored.csv", index=False)
    summary.to_csv("assignment_outputs/self_summary.csv", index=False)
    print("Saved self results to assignment_outputs/")

### Part C: Measure the MODEL's Calibration (15 points)

Query an LLM (use the OpenAI API, Anthropic API, or any accessible API — free tiers are fine) with all 50 questions. For each question, use the following prompt template:

```
Answer the following question. After your answer, rate your confidence
from 0% to 100% that your answer is correct. Format your response as:
Answer: [your answer]
Confidence: [X]%
```

Then:
1. **Parse** the model's answers and confidence ratings
2. **Score** each answer as correct or incorrect (you may need fuzzy matching — document your approach)
3. Compute the **model's calibration** the same way you computed yours:
   - Accuracy per difficulty category
   - Average confidence per difficulty category
   - Calibration error per category
   - Calibration plot

4. **Compare** your calibration plot to the model's calibration plot side by side.

**Key analysis questions** (answer in a markdown cell):
- Is the model overconfident, underconfident, or well-calibrated?
- Does its calibration error vary with difficulty? (Compare to humans from Lichtenstein et al. 1982)
- How does the model handle the unanswerable questions? Does it say "I don't know" or does it confabulate confidently?
- How does the model's calibration pattern differ from yours?

In [ ]:
# Part C: Measure MODEL calibration (baseline)
# OpenRouter setup: uses OpenAI-compatible client with OpenRouter base URL.

import os

if "df_questions" not in globals():
    df_questions = pd.read_csv("assignment_outputs/questions.csv")

BASE_PROMPT = (
    "Answer the following question. After your answer, rate your confidence\n"
    "from 0% to 100% that your answer is correct. Format your response as:\n"
    "Answer: [your answer]\n"
    "Confidence: [X]%"
)

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_DEFAULT_MODEL = os.environ.get("OPENROUTER_MODEL", "openai/gpt-4.1-mini")


def parse_answer_confidence(text: str):
    answer_match = re.search(r"Answer\s*:\s*(.+?)(?:\n|$)", text, flags=re.IGNORECASE | re.DOTALL)
    conf_match = re.search(r"Confidence\s*:\s*([0-9]{1,3})(?:\s*%)?", text, flags=re.IGNORECASE)

    answer = answer_match.group(1).strip() if answer_match else text.strip().split("\n")[0]
    if conf_match:
        confidence = float(conf_match.group(1))
    else:
        confidence = np.nan
    confidence = np.clip(confidence, 0, 100) if not np.isnan(confidence) else np.nan
    return answer, confidence


def get_openrouter_client():
    from openai import OpenAI

    api_key = os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        raise EnvironmentError("OPENROUTER_API_KEY not found.")

    return OpenAI(
        api_key=api_key,
        base_url=OPENROUTER_BASE_URL,
    )


def query_openrouter_for_questions(df, model=OPENROUTER_DEFAULT_MODEL, temperature=0.2, system_prompt="You are a concise factual assistant."):
    client = get_openrouter_client()
    rows = []

    for i, row in df.iterrows():
        user_msg = f"{BASE_PROMPT}\n\nQuestion: {row['question']}"
        resp = client.chat.completions.create(
            model=model,
            temperature=temperature,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_msg},
            ],
            extra_headers={
                "HTTP-Referer": os.environ.get("OPENROUTER_SITE_URL", "http://localhost"),
                "X-Title": os.environ.get("OPENROUTER_APP_NAME", "NST2062 Assignment"),
            },
        )
        raw = resp.choices[0].message.content
        ans, conf = parse_answer_confidence(raw)
        rows.append(
            {
                "question": row["question"],
                "ground_truth": row["answer"],
                "difficulty": row["difficulty"],
                "raw_response": raw,
                "model_answer": ans,
                "confidence": conf,
                "model": model,
                "temperature": temperature,
            }
        )
        if (i + 1) % 10 == 0:
            print(f"Completed {i + 1}/{len(df)}")

    return pd.DataFrame(rows)


baseline_path = Path("assignment_outputs/model_baseline_outputs.csv")

if baseline_path.exists():
    model_df = pd.read_csv(baseline_path)
    print("Loaded cached baseline outputs:", baseline_path)
else:
    if not os.environ.get("OPENROUTER_API_KEY"):
        raise EnvironmentError(
            "OPENROUTER_API_KEY not found and no cached baseline CSV found. "
            "Set OPENROUTER_API_KEY (and optionally OPENROUTER_MODEL) first."
        )
    model_df = query_openrouter_for_questions(df_questions, model=OPENROUTER_DEFAULT_MODEL, temperature=0.2)
    model_df.to_csv(baseline_path, index=False)
    print("Saved baseline outputs:", baseline_path)

model_df["correct"] = model_df.apply(lambda r: is_correct_answer(r["model_answer"], r["ground_truth"]), axis=1)

model_summary = (
    model_df.groupby("difficulty", as_index=False)
    .agg(
        accuracy=("correct", "mean"),
        avg_confidence=("confidence", "mean"),
        n=("question", "size"),
    )
)
model_summary["calibration_error_abs"] = (model_summary["avg_confidence"] / 100.0 - model_summary["accuracy"]).abs()

print("Model summary by difficulty")
display(model_summary)
print("Model ECE:", expected_calibration_error(model_df, conf_col="confidence", correct_col="correct", n_bins=10))

self_path = Path("assignment_outputs/self_results_scored.csv")
if self_path.exists():
    self_scored = pd.read_csv(self_path)

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    bucket_self = calibration_by_bucket(
        self_scored.rename(columns={"my_confidence": "confidence"}),
        conf_col="confidence",
        correct_col="correct",
        n_bins=10,
    )
    plt.plot([0, 1], [0, 1], "k--")
    if not bucket_self.empty:
        plt.plot(bucket_self["avg_confidence"], bucket_self["accuracy"], marker="o", label="You")
    plt.title("Your Calibration")
    plt.xlabel("Mean confidence")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    bucket_model = calibration_by_bucket(model_df, conf_col="confidence", correct_col="correct", n_bins=10)
    plt.plot([0, 1], [0, 1], "k--")
    if not bucket_model.empty:
        plt.plot(bucket_model["avg_confidence"], bucket_model["accuracy"], marker="o", label="Model", color="tab:orange")
    plt.title("Model Calibration")
    plt.xlabel("Mean confidence")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Self-scored file not found; skipping side-by-side plot.")

model_df.to_csv("assignment_outputs/model_baseline_scored.csv", index=False)
model_summary.to_csv("assignment_outputs/model_baseline_summary.csv", index=False)
print("Saved model baseline scored outputs to assignment_outputs/")

### Part D: Can You Improve the Model's Calibration? (10 points)

Try at least **two** prompt engineering interventions to improve the model's calibration. Ideas:
- Ask it to "think step by step before rating confidence"
- Ask it to "list reasons it might be wrong before answering"
- Ask it to "rate confidence on a scale of 1-10 instead of percentage"
- Provide a few-shot example of well-calibrated answers (including "I'm not sure")
- Ask in a different persona ("You are a careful scientist who hates being wrong...")

For each intervention:
1. Re-run the 50 questions with the new prompt
2. Compute calibration again
3. Compare to the baseline

**Report**: Which intervention helped most? Which didn't help? Why do you think that is (connect to what you know about RLHF and reward signals from Lecture 3)?

In [ ]:
# Part D: Prompt engineering interventions for calibration

import os

if "df_questions" not in globals():
    df_questions = pd.read_csv("assignment_outputs/questions.csv")

if "query_openrouter_for_questions" not in globals():
    raise RuntimeError("Run Part C cell first to define OpenRouter query helpers.")

interventions = {
    "baseline": "You are a concise factual assistant.",
    "reason_before_confidence": (
        "You are a careful assistant. Think through the evidence briefly before answering. "
        "Then provide confidence conservatively."
    ),
    "list_why_wrong": (
        "You are a careful scientist who hates being wrong. Before final confidence, "
        "consider at least two reasons your answer might be wrong."
    ),
}

all_runs = []
for name, sys_prompt in interventions.items():
    run_path = Path(f"assignment_outputs/model_{name}_outputs.csv")

    if run_path.exists():
        run_df = pd.read_csv(run_path)
        print(f"Loaded cached intervention run: {name}")
    else:
        if not os.environ.get("OPENROUTER_API_KEY"):
            print(f"Skipping {name}: no OPENROUTER_API_KEY and no cached file.")
            continue
        run_df = query_openrouter_for_questions(
            df_questions,
            model=OPENROUTER_DEFAULT_MODEL,
            temperature=0.2,
            system_prompt=sys_prompt,
        )
        run_df.to_csv(run_path, index=False)
        print(f"Saved intervention run: {run_path}")

    run_df["intervention"] = name
    run_df["correct"] = run_df.apply(lambda r: is_correct_answer(r["model_answer"], r["ground_truth"]), axis=1)
    all_runs.append(run_df)

if not all_runs:
    raise RuntimeError("No intervention runs available. Provide OPENROUTER_API_KEY or cached CSV files.")

intervention_df = pd.concat(all_runs, ignore_index=True)
intervention_df.to_csv("assignment_outputs/interventions_all_scored.csv", index=False)

summary = (
    intervention_df.groupby(["intervention", "difficulty"], as_index=False)
    .agg(
        accuracy=("correct", "mean"),
        avg_confidence=("confidence", "mean"),
        n=("question", "size"),
    )
)
summary["calibration_error_abs"] = (summary["avg_confidence"] / 100.0 - summary["accuracy"]).abs()

overall = (
    intervention_df.groupby("intervention", as_index=False)
    .apply(lambda g: pd.Series({
        "ece": expected_calibration_error(g, conf_col="confidence", correct_col="correct", n_bins=10),
        "accuracy": g["correct"].mean(),
        "avg_confidence": g["confidence"].mean(),
    }), include_groups=False)
    .reset_index(drop=True)
)

print("Intervention summary by difficulty")
display(summary)
print("Overall intervention performance")
display(overall.sort_values("ece"))

plt.figure(figsize=(8, 4))
sns.barplot(data=overall.sort_values("ece"), x="intervention", y="ece")
plt.title("Calibration Error (ECE) by Intervention")
plt.ylabel("ECE (lower is better)")
plt.xlabel("Intervention")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 5))
for _, row in overall.iterrows():
    plt.scatter(row["avg_confidence"] / 100.0, row["accuracy"], s=120, label=row["intervention"])
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("Average confidence")
plt.ylabel("Accuracy")
plt.title("Confidence vs Accuracy by Intervention")
plt.legend()
plt.tight_layout()
plt.show()

summary.to_csv("assignment_outputs/interventions_summary_by_difficulty.csv", index=False)
overall.to_csv("assignment_outputs/interventions_overall.csv", index=False)
print("Saved intervention summaries to assignment_outputs/")

### Bonus: Token-Level Probabilities (10 bonus points)

If you're using an API that exposes log-probabilities (e.g., OpenAI with `logprobs=True`):
1. Extract the **token-level probabilities** for the model's answers
2. Compare the model's **stated** confidence (from the text) to its **actual** token probabilities
3. Are they correlated? Is the model's internal uncertainty (log-probs) better calibrated than its stated confidence?

This gets at a deep question from Lecture 2 (Representation): the model may have an internal representation of uncertainty that differs from what it expresses in text. The expressed confidence was shaped by RLHF; the log-probs were shaped by pre-training.

In [ ]:
# Bonus: Token-level probabilities vs stated confidence
# This requires an OpenRouter model/provider that returns chat logprobs.

import os

if "df_questions" not in globals():
    df_questions = pd.read_csv("assignment_outputs/questions.csv")

if "get_openrouter_client" not in globals():
    raise RuntimeError("Run Part C first to initialize OpenRouter helpers.")


def query_with_logprobs(df, model=OPENROUTER_DEFAULT_MODEL, temperature=0.2):
    client = get_openrouter_client()
    rows = []

    for i, row in df.iterrows():
        user_msg = f"{BASE_PROMPT}\n\nQuestion: {row['question']}"
        resp = client.chat.completions.create(
            model=model,
            temperature=temperature,
            logprobs=True,
            top_logprobs=5,
            messages=[
                {"role": "system", "content": "You are a concise factual assistant."},
                {"role": "user", "content": user_msg},
            ],
            extra_headers={
                "HTTP-Referer": os.environ.get("OPENROUTER_SITE_URL", "http://localhost"),
                "X-Title": os.environ.get("OPENROUTER_APP_NAME", "NST2062 Assignment"),
            },
        )

        raw = resp.choices[0].message.content
        ans, conf = parse_answer_confidence(raw)

        token_logprobs = []
        try:
            content = resp.choices[0].logprobs.content
            token_logprobs = [t.logprob for t in content if t.logprob is not None]
        except Exception:
            token_logprobs = []

        mean_token_prob = float(np.exp(np.mean(token_logprobs))) if token_logprobs else np.nan

        rows.append(
            {
                "question": row["question"],
                "ground_truth": row["answer"],
                "difficulty": row["difficulty"],
                "raw_response": raw,
                "model_answer": ans,
                "stated_confidence": conf,
                "mean_token_prob": mean_token_prob,
            }
        )

        if (i + 1) % 10 == 0:
            print(f"Completed {i + 1}/{len(df)}")

    return pd.DataFrame(rows)


bonus_path = Path("assignment_outputs/model_logprobs_outputs.csv")

if bonus_path.exists():
    bonus_df = pd.read_csv(bonus_path)
    print("Loaded cached logprobs output")
else:
    if not os.environ.get("OPENROUTER_API_KEY"):
        raise EnvironmentError("OPENROUTER_API_KEY missing and no cached logprobs output available.")
    bonus_df = query_with_logprobs(df_questions)
    bonus_df.to_csv(bonus_path, index=False)

bonus_df["correct"] = bonus_df.apply(lambda r: is_correct_answer(r["model_answer"], r["ground_truth"]), axis=1)
bonus_df = bonus_df.dropna(subset=["stated_confidence", "mean_token_prob"]).copy()

if bonus_df.empty:
    print("No valid rows with both stated confidence and token probabilities.")
else:
    corr = bonus_df["stated_confidence"].corr(bonus_df["mean_token_prob"])
    print(f"Correlation between stated confidence and mean token probability: {corr:.3f}")

    plt.figure(figsize=(6, 5))
    sns.scatterplot(data=bonus_df, x="stated_confidence", y="mean_token_prob", hue="correct")
    plt.title("Stated Confidence vs Mean Token Probability")
    plt.xlabel("Stated confidence (%)")
    plt.ylabel("Mean token probability (exp(mean logprob))")
    plt.tight_layout()
    plt.show()

    by_correct = bonus_df.groupby("correct", as_index=False).agg(
        mean_stated_confidence=("stated_confidence", "mean"),
        mean_token_prob=("mean_token_prob", "mean"),
        n=("question", "size"),
    )
    display(by_correct)

bonus_df.to_csv("assignment_outputs/model_logprobs_scored.csv", index=False)
print("Saved bonus analysis outputs.")

### 🪞 Metacognitive Reflection — Problem 1 (graded, 5 points included above)

Write 200–400 words answering these questions. **This must be in your own voice — not generated by an LLM.**

1. Were you surprised by your own calibration results? Where were you most overconfident?
2. Were you surprised by the model's calibration? How did it compare to your expectations?
3. What did you learn about the relationship between confidence and accuracy — in yourself and in the model?
4. How did you use LLMs during this problem? What did you prompt for, what did you have to fix, and what did the LLM get wrong?
5. Connect your findings to one concept from the course (e.g., Goodhart's Law, prediction error, the Bayesian brain). How does your data illustrate or challenge that concept?

*Your reflection here (double-click to edit):*




---
## Problem 2: Temperature, Creativity, and Mode Collapse — Does Optimization Kill Novelty?
### (35 points + 10 bonus)

**Course connections**: Creativity (mode collapse, exploration vs exploitation), Learning & Reward (RLHF as reward shaping), Representation (traversal of embedding space), Decisions (temperature as the creativity dial).

**The question**: In Lecture 5 (Creativity), we argued that RLHF causes mode collapse — the model converges on safe, crowd-pleasing outputs and loses the tail distribution where novelty lives. In this problem, you'll measure this directly.

---

### Part A: The Divergent Thinking Experiment (10 points)

Use the **Alternative Uses Task** (Guilford, 1967) — the same test you did in the Creativity lecture.

Pick **3 common objects** (e.g., brick, paperclip, shoe). For each object, prompt the LLM to:
> "List 20 unusual uses for a [object]. Be creative and surprising."

Run this at **5 different temperature settings**: 0.0, 0.3, 0.7, 1.0, 1.5 (or as close as the API allows).

For each (object × temperature) combination, record all 20 responses.

That gives you 3 objects × 5 temperatures × 20 uses = 300 data points.

In [ ]:
# Problem 2, Part A: Divergent thinking across temperatures

import os

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_DEFAULT_MODEL = os.environ.get("OPENROUTER_MODEL", "openai/gpt-4.1-mini")

objects = ["brick", "paperclip", "shoe"]
temperatures = [0.0, 0.3, 0.7, 1.0, 1.5]

PROMPT_TEMPLATE = "List 20 unusual uses for a {obj}. Be creative and surprising. Return as a numbered list."


def parse_numbered_list(text: str):
    lines = [l.strip() for l in str(text).splitlines() if l.strip()]
    uses = []
    for line in lines:
        line = re.sub(r"^\d+[\).:-]?\s*", "", line)
        if line:
            uses.append(line)
    return uses[:20]


def get_openrouter_client_local():
    from openai import OpenAI

    api_key = os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        raise EnvironmentError("OPENROUTER_API_KEY not found.")

    return OpenAI(api_key=api_key, base_url=OPENROUTER_BASE_URL)


def run_divergent_experiment(objects, temperatures, model=OPENROUTER_DEFAULT_MODEL):
    client = get_openrouter_client_local()
    rows = []

    for obj in objects:
        for temp in temperatures:
            prompt = PROMPT_TEMPLATE.format(obj=obj)
            resp = client.chat.completions.create(
                model=model,
                temperature=float(temp),
                messages=[
                    {"role": "system", "content": "You are a creative but concise assistant."},
                    {"role": "user", "content": prompt},
                ],
                extra_headers={
                    "HTTP-Referer": os.environ.get("OPENROUTER_SITE_URL", "http://localhost"),
                    "X-Title": os.environ.get("OPENROUTER_APP_NAME", "NST2062 Assignment"),
                },
            )
            raw = resp.choices[0].message.content
            uses = parse_numbered_list(raw)

            for i in range(20):
                use_text = uses[i] if i < len(uses) else ""
                rows.append(
                    {
                        "object": obj,
                        "temperature": temp,
                        "item_index": i + 1,
                        "use_text": use_text,
                        "raw_response": raw,
                    }
                )
            print(f"Done object={obj}, temp={temp}")

    return pd.DataFrame(rows)


out_path = Path("assignment_outputs/divergent_uses.csv")
if out_path.exists():
    divergent_df = pd.read_csv(out_path)
    print("Loaded cached divergent uses:", out_path)
else:
    if not os.environ.get("OPENROUTER_API_KEY"):
        raise EnvironmentError("OPENROUTER_API_KEY missing and no cached divergent_uses.csv found.")
    divergent_df = run_divergent_experiment(objects, temperatures)
    divergent_df.to_csv(out_path, index=False)
    print("Saved:", out_path)

print("Rows:", len(divergent_df))
display(divergent_df.head(12))
print("Expected 300 rows for complete data (3 objects x 5 temps x 20 uses).")

### Part B: Measuring Creativity — Design Your Own Rubric (10 points)

This is the hard part. There is no standard metric for creativity (that was the Evaluation Problem from Lecture 5). You need to **design a scoring rubric** with at least 3 dimensions. Suggested dimensions (pick 3-4, or invent your own):

- **Originality**: How surprising/uncommon is the use? (1 = obvious, 5 = never heard of it)
- **Feasibility**: Could you actually do this? (1 = impossible, 5 = easy)
- **Humour**: Is it funny or playful? (1 = not at all, 5 = genuinely funny)
- **Elaboration**: How detailed/specific is the response? (1 = generic, 5 = vivid)
- **Category diversity**: Does the list span different domains (physical, social, artistic, absurd)?

**Important**: Score a random sample of **5 uses per temperature** per object (75 total). Do this yourself — do NOT use an LLM to evaluate creativity.

Then compute and plot:
1. **Average score per dimension** as a function of temperature
2. **Lexical diversity** (number of unique words / total words) as a function of temperature
3. **Semantic diversity** (average pairwise cosine distance between sentence embeddings) as a function of temperature

For semantic diversity, use a sentence embedding model (e.g., `sentence-transformers/all-MiniLM-L6-v2` from HuggingFace).

In [ ]:
# Problem 2, Part B: Creativity rubric + diversity metrics

from collections import Counter
from itertools import combinations

if "divergent_df" not in globals():
    divergent_df = pd.read_csv("assignment_outputs/divergent_uses.csv")

# 1) Manual rubric scoring template (5 uses per temperature per object => 75 rows)
rubric_path = Path("assignment_outputs/creativity_rubric_scores.csv")

sample_df = (
    divergent_df.groupby(["object", "temperature"], group_keys=False)
    .apply(lambda x: x.sample(5, random_state=123))
    .reset_index(drop=True)
)

if rubric_path.exists():
    rubric_df = pd.read_csv(rubric_path)
else:
    rubric_df = sample_df[["object", "temperature", "item_index", "use_text"]].copy()
    rubric_df["originality"] = np.nan
    rubric_df["feasibility"] = np.nan
    rubric_df["humour"] = np.nan
    rubric_df["elaboration"] = np.nan
    rubric_df.to_csv(rubric_path, index=False)

print("Fill rubric scores (1-5) in:", rubric_path)
display(rubric_df.head(10))

rubric_df = pd.read_csv(rubric_path)
if rubric_df[["originality", "feasibility", "humour", "elaboration"]].isna().any().any():
    print("Please fill all rubric values (1-5), then re-run this cell.")
else:
    rubric_long = rubric_df.melt(
        id_vars=["object", "temperature", "item_index", "use_text"],
        value_vars=["originality", "feasibility", "humour", "elaboration"],
        var_name="dimension",
        value_name="score",
    )

    rubric_summary = (
        rubric_long.groupby(["temperature", "dimension"], as_index=False)
        .agg(avg_score=("score", "mean"), n=("score", "size"))
    )

    plt.figure(figsize=(9, 5))
    sns.lineplot(data=rubric_summary, x="temperature", y="avg_score", hue="dimension", marker="o")
    plt.title("Average Creativity Rubric Scores vs Temperature")
    plt.ylim(1, 5)
    plt.tight_layout()
    plt.show()


# 2) Lexical diversity (unique words / total words)
def lexical_diversity(texts):
    tokens = []
    for t in texts:
        tokens += re.findall(r"[a-zA-Z']+", str(t).lower())
    if not tokens:
        return np.nan
    return len(set(tokens)) / len(tokens)

lexical = (
    divergent_df.groupby("temperature", as_index=False)
    .apply(lambda g: pd.Series({"lexical_diversity": lexical_diversity(g["use_text"])}), include_groups=False)
    .reset_index(drop=True)
)

plt.figure(figsize=(7, 4))
sns.lineplot(data=lexical, x="temperature", y="lexical_diversity", marker="o")
plt.title("Lexical Diversity vs Temperature")
plt.tight_layout()
plt.show()


# 3) Semantic diversity (avg pairwise cosine distance)
# Preferred: sentence-transformers embeddings. Fallback: TF-IDF vectors if unavailable.
def semantic_diversity_for_group(texts):
    texts = [str(t) for t in texts if str(t).strip()]
    if len(texts) < 2:
        return np.nan

    try:
        from sentence_transformers import SentenceTransformer
        from sklearn.metrics.pairwise import cosine_similarity

        model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        emb = model.encode(texts, show_progress_bar=False)
        sim = cosine_similarity(emb)
    except Exception:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.metrics.pairwise import cosine_similarity

        vec = TfidfVectorizer(stop_words="english")
        X = vec.fit_transform(texts)
        sim = cosine_similarity(X)

    i_upper = np.triu_indices_from(sim, k=1)
    pairwise_dist = 1 - sim[i_upper]
    return float(np.mean(pairwise_dist)) if len(pairwise_dist) else np.nan

semantic = (
    divergent_df.groupby("temperature", as_index=False)
    .apply(lambda g: pd.Series({"semantic_diversity": semantic_diversity_for_group(g["use_text"])}), include_groups=False)
    .reset_index(drop=True)
)

plt.figure(figsize=(7, 4))
sns.lineplot(data=semantic, x="temperature", y="semantic_diversity", marker="o", color="tab:green")
plt.title("Semantic Diversity vs Temperature")
plt.tight_layout()
plt.show()


display(lexical)
display(semantic)
lexical.to_csv("assignment_outputs/lexical_diversity_by_temp.csv", index=False)
semantic.to_csv("assignment_outputs/semantic_diversity_by_temp.csv", index=False)

if 'rubric_summary' in locals():
    rubric_summary.to_csv("assignment_outputs/rubric_summary_by_temp.csv", index=False)

print("Saved creativity metrics to assignment_outputs/")

### Part C: Detecting Mode Collapse (15 points)

Now test whether RLHF narrows the output distribution. Design an experiment:

1. **Repetition test**: Run the same prompt 10 times at temperature 0.7. How many of the 20 uses are **identical** across runs? How many are **semantically similar** (cosine similarity > 0.85)? A model in mode collapse will repeat itself; a diverse model won't.

2. **Safe vs. surprising**: Classify each use as **safe** (conventional, predictable — e.g., "use a brick as a doorstop") or **surprising** (unexpected, creative — e.g., "use a brick as a pillow to build character"). Plot the ratio of safe:surprising as a function of temperature.

3. **Compare models (if possible)**: If you have access to both a base model and an RLHF'd model (e.g., via different API endpoints or open-source models like Llama base vs. Llama-chat), compare their diversity at the same temperature. The prediction from Lecture 5: the RLHF'd model should be less diverse.

**Analysis questions** (answer in a markdown cell):
- At what temperature does the model produce the best balance of quality and originality?
- Is there evidence of mode collapse? Where?
- How does this connect to the exploration/exploitation tradeoff from the Decisions lecture?
- If you compared models: does RLHF reduce diversity as predicted?

In [ ]:
# Problem 2, Part C: Mode collapse detection

import os

if "divergent_df" not in globals():
    divergent_df = pd.read_csv("assignment_outputs/divergent_uses.csv")

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_DEFAULT_MODEL = os.environ.get("OPENROUTER_MODEL", "openai/gpt-4.1-mini")

# 1) Repetition test: repeat same prompt 10 times at temperature 0.7
repetition_path = Path("assignment_outputs/repetition_runs_temp07.csv")


def get_openrouter_client_local():
    from openai import OpenAI

    api_key = os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        raise EnvironmentError("OPENROUTER_API_KEY not found.")

    return OpenAI(api_key=api_key, base_url=OPENROUTER_BASE_URL)


def run_repetition_test(obj="brick", n_runs=10, model=OPENROUTER_DEFAULT_MODEL, temp=0.7):
    client = get_openrouter_client_local()
    rows = []
    prompt = f"List 20 unusual uses for a {obj}. Be creative and surprising. Return as a numbered list."

    for run_id in range(1, n_runs + 1):
        resp = client.chat.completions.create(
            model=model,
            temperature=temp,
            messages=[
                {"role": "system", "content": "You are a creative but concise assistant."},
                {"role": "user", "content": prompt},
            ],
            extra_headers={
                "HTTP-Referer": os.environ.get("OPENROUTER_SITE_URL", "http://localhost"),
                "X-Title": os.environ.get("OPENROUTER_APP_NAME", "NST2062 Assignment"),
            },
        )
        raw = resp.choices[0].message.content
        uses = parse_numbered_list(raw)
        for i in range(20):
            rows.append(
                {
                    "run_id": run_id,
                    "object": obj,
                    "temperature": temp,
                    "item_index": i + 1,
                    "use_text": uses[i] if i < len(uses) else "",
                }
            )
        print(f"Completed repetition run {run_id}/{n_runs}")

    return pd.DataFrame(rows)


if repetition_path.exists():
    repetition_df = pd.read_csv(repetition_path)
    print("Loaded cached repetition runs")
else:
    if not os.environ.get("OPENROUTER_API_KEY"):
        raise EnvironmentError("OPENROUTER_API_KEY missing and no cached repetition runs found.")
    repetition_df = run_repetition_test(obj="brick", n_runs=10, temp=0.7)
    repetition_df.to_csv(repetition_path, index=False)

norm_text = repetition_df["use_text"].fillna("").map(normalize_text)
exact_dup_rate = 1 - (norm_text.nunique() / len(norm_text))
print(f"Exact duplicate rate: {exact_dup_rate:.3f}")

texts = repetition_df["use_text"].fillna("").tolist()
try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity

    emb_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    emb = emb_model.encode(texts, show_progress_bar=False)
    sim = cosine_similarity(emb)
except Exception:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

    X = TfidfVectorizer(stop_words="english").fit_transform(texts)
    sim = cosine_similarity(X)

upper = np.triu_indices_from(sim, k=1)
similar_pairs = (sim[upper] > 0.85).mean()
print(f"Pairwise semantic-similarity>0.85 proportion: {similar_pairs:.3f}")

safe_path = Path("assignment_outputs/safe_surprising_labels.csv")
if safe_path.exists():
    safe_df = pd.read_csv(safe_path)
else:
    safe_df = divergent_df[["object", "temperature", "item_index", "use_text"]].copy()
    safe_df["label"] = ""
    safe_df.to_csv(safe_path, index=False)

print("Label outputs as safe/surprising in:", safe_path)

safe_df = pd.read_csv(safe_path)
if safe_df["label"].astype(str).str.strip().eq("").any():
    print("Please fill all labels with 'safe' or 'surprising', then re-run this cell.")
else:
    safe_df["label"] = safe_df["label"].str.strip().str.lower()
    valid = safe_df["label"].isin(["safe", "surprising"])
    if not valid.all():
        bad = safe_df.loc[~valid, "label"].unique()
        raise ValueError(f"Invalid labels found: {bad}. Use only 'safe' or 'surprising'.")

    ratio = (
        safe_df.groupby(["temperature", "label"], as_index=False)
        .size()
        .pivot(index="temperature", columns="label", values="size")
        .fillna(0)
        .reset_index()
    )
    ratio["safe_to_surprising"] = ratio.get("safe", 0) / ratio.get("surprising", 1)

    plt.figure(figsize=(8, 4))
    sns.lineplot(data=ratio, x="temperature", y="safe_to_surprising", marker="o")
    plt.title("Safe:Surprising Ratio vs Temperature")
    plt.ylabel("safe / surprising")
    plt.tight_layout()
    plt.show()

    display(ratio)
    ratio.to_csv("assignment_outputs/safe_surprising_ratio_by_temp.csv", index=False)

print("Mode collapse analysis files saved in assignment_outputs/")

### Bonus: The Audience Matters (10 bonus points)

**Course connection**: Mentalising (Lecture 6) — creativity needs an audience.

Add audience context to the prompt and measure whether it changes the outputs:
> "List 20 unusual uses for a brick. Your audience is [a group of 5-year-olds / a panel of art critics / a team of engineers / a comedy club audience]."

For each audience:
1. Run at temperature 0.7
2. Score using your rubric
3. Measure how the output distribution shifts

**Analysis**: Does the model adapt its creativity to the audience? Is it doing something like mentalising — modelling what each audience would find creative? Or is it just pattern-matching on audience-associated vocabulary?

In [ ]:
# Bonus: Audience-context creativity experiment

import os

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_DEFAULT_MODEL = os.environ.get("OPENROUTER_MODEL", "openai/gpt-4.1-mini")

audiences = [
    "a group of 5-year-olds",
    "a panel of art critics",
    "a team of engineers",
    "a comedy club audience",
]

bonus_audience_path = Path("assignment_outputs/audience_experiment.csv")


def get_openrouter_client_local():
    from openai import OpenAI

    api_key = os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        raise EnvironmentError("OPENROUTER_API_KEY not found.")

    return OpenAI(api_key=api_key, base_url=OPENROUTER_BASE_URL)


def run_audience_experiment(object_name="brick", audiences=None, temperature=0.7, model=OPENROUTER_DEFAULT_MODEL):
    client = get_openrouter_client_local()
    rows = []

    for audience in audiences:
        prompt = (
            f"List 20 unusual uses for a {object_name}. "
            f"Your audience is {audience}. Be creative and surprising. Return as a numbered list."
        )
        resp = client.chat.completions.create(
            model=model,
            temperature=temperature,
            messages=[
                {"role": "system", "content": "You adapt creativity to the audience while staying coherent."},
                {"role": "user", "content": prompt},
            ],
            extra_headers={
                "HTTP-Referer": os.environ.get("OPENROUTER_SITE_URL", "http://localhost"),
                "X-Title": os.environ.get("OPENROUTER_APP_NAME", "NST2062 Assignment"),
            },
        )

        raw = resp.choices[0].message.content
        uses = parse_numbered_list(raw)

        for i in range(20):
            rows.append(
                {
                    "object": object_name,
                    "audience": audience,
                    "temperature": temperature,
                    "item_index": i + 1,
                    "use_text": uses[i] if i < len(uses) else "",
                    "raw_response": raw,
                }
            )
        print(f"Completed audience: {audience}")

    return pd.DataFrame(rows)


if bonus_audience_path.exists():
    audience_df = pd.read_csv(bonus_audience_path)
    print("Loaded cached audience experiment")
else:
    if not os.environ.get("OPENROUTER_API_KEY"):
        raise EnvironmentError("OPENROUTER_API_KEY missing and no cached audience_experiment.csv found.")
    audience_df = run_audience_experiment(object_name="brick", audiences=audiences, temperature=0.7)
    audience_df.to_csv(bonus_audience_path, index=False)

audience_rubric_path = Path("assignment_outputs/audience_rubric_scores.csv")
if audience_rubric_path.exists():
    audience_rubric = pd.read_csv(audience_rubric_path)
else:
    audience_rubric = audience_df[["audience", "item_index", "use_text"]].copy()
    audience_rubric = audience_rubric.groupby("audience", group_keys=False).apply(lambda g: g.sample(5, random_state=7)).reset_index(drop=True)
    audience_rubric["originality"] = np.nan
    audience_rubric["feasibility"] = np.nan
    audience_rubric["humour"] = np.nan
    audience_rubric["elaboration"] = np.nan
    audience_rubric.to_csv(audience_rubric_path, index=False)

print("If doing bonus scoring, fill:", audience_rubric_path)

audience_rubric = pd.read_csv(audience_rubric_path)
if not audience_rubric[["originality", "feasibility", "humour", "elaboration"]].isna().any().any():
    aud_summary = (
        audience_rubric.groupby("audience", as_index=False)
        .agg(
            originality=("originality", "mean"),
            feasibility=("feasibility", "mean"),
            humour=("humour", "mean"),
            elaboration=("elaboration", "mean"),
        )
    )
    display(aud_summary)

    aud_long = aud_summary.melt(id_vars="audience", var_name="dimension", value_name="score")
    plt.figure(figsize=(10, 5))
    sns.barplot(data=aud_long, x="audience", y="score", hue="dimension")
    plt.title("Audience-Conditioned Creativity Scores")
    plt.xticks(rotation=20)
    plt.ylim(1, 5)
    plt.tight_layout()
    plt.show()

def lex_div(texts):
    toks = []
    for t in texts:
        toks.extend(re.findall(r"[a-zA-Z']+", str(t).lower()))
    return (len(set(toks)) / len(toks)) if toks else np.nan

lex_by_aud = (
    audience_df.groupby("audience", as_index=False)
    .apply(lambda g: pd.Series({"lexical_diversity": lex_div(g["use_text"])}), include_groups=False)
    .reset_index(drop=True)
)
display(lex_by_aud)
lex_by_aud.to_csv("assignment_outputs/audience_lexical_diversity.csv", index=False)
print("Saved audience experiment outputs to assignment_outputs/")

### 🪞 Metacognitive Reflection — Problem 2 (graded, 5 points included above)

Write 200–400 words answering these questions. **This must be in your own voice — not generated by an LLM.**

1. What was the hardest part of designing a creativity rubric? What did it teach you about the evaluation problem?
2. Did the temperature results match your predictions? What surprised you?
3. Did you find evidence of mode collapse? If yes, is this a problem — or is it the model being appropriately safe?
4. How did you use LLMs during this problem? Be specific about what you prompted for and what you had to do yourself.
5. Connect your findings to the course: if creativity is search through representation space, and temperature controls the search width, what does your data tell you about the shape of that space?

*Your reflection here (double-click to edit):*




---
## Grading Summary

| Component | Points |
|---|---|
| **Problem 1** | |
| Part A: Calibration dataset | 10 |
| Part B: Your own calibration | 10 |
| Part C: Model calibration + comparison | 15 |
| Part D: Prompt interventions | 10 |
| Bonus: Token-level probabilities | (+10) |
| **Problem 2** | |
| Part A: Divergent thinking experiment | 10 |
| Part B: Creativity rubric + metrics | 10 |
| Part C: Mode collapse detection | 15 |
| Bonus: Audience experiment | (+10) |
| **Metacognitive Reflections** | |
| Reflection 1 (included in P1) | (5 of P1) |
| Reflection 2 (included in P2) | (5 of P2) |
| **Total** | **80 + 20 bonus** |

---

### Assessment Criteria

**Code quality** (30%): Does the code run? Is it clean and documented? Are the experiments reproducible?

**Analysis quality** (40%): Are the visualisations clear? Are the comparisons meaningful? Do the analysis questions receive thoughtful, evidence-based answers that connect to course concepts?

**Metacognitive reflections** (20%): Are the reflections genuine, specific, and self-aware? Do they demonstrate learning — not just completion?

**Rigour** (10%): Sample sizes, statistical thinking, awareness of limitations. You don't need p-values, but you should know when your sample is too small to draw strong conclusions.

---

### A Note on Using LLMs

You will use LLMs both as a **tool** (to help you code) and as the **object of study** (the thing you're investigating). This is intentional. The metacognitive reflections ask you to distinguish between these two roles. The ability to use a tool critically while simultaneously studying it is a core skill for working with AI systems — and it's a form of mentalising: modelling the capabilities and limitations of a non-human agent you're interacting with.

Have fun! I am looking forward to see your results!